# Computational Analysis of Sounds and Music (CH-CASM-M)

## 06 - Source Separation

**WS 2025/2026**

Prof. Dr. Jakob Abeßer, jakob.abesser@uni-bamberg.de

Last update: 25.11.2025

**Outline**

In this notebook, we will study the two signal-processing based source separation algorithms
- Harmonic/Percussive Source Separation (HPSS)
- Non-Negative Matrix Factorization (NMF)

## Preparation

In [ ]:
!pip install wget

In [ ]:
import glob
import os
import librosa
import librosa.display
import numpy as np
import wget
import matplotlib.pyplot as pl
import IPython.display as ipd

## Audio Files

We use in this notebook several audio files taken from the Free Sound library (https://freesound.org/).

In [ ]:
fn_wav_list = ['196765__xserra__piano-phrase.wav',
               '257993__orangefreesounds__disco-funky-beat.wav',
               '641823__szegvari__drumjam-conga-solo-sample-ethno-music-drums-119bpm_2022-07-15_191240_CUT.wav']

if any([not os.path.isfile(_) for _ in fn_wav_list]):
    for fn in fn_wav_list:
        wget.download('https://github.com/CHBamberg/CH-CASM-M-2025/raw/refs/heads/main/data/{}'.format(fn), 
                      out=fn, bar=None)
else:
    print('Files already exist!')

## Harmonic/Percussive Source Separation

Some helper functions ...

In [ ]:
def audio_playback_hpss(y, y_h, y_p, sr):
    print("Original")
    ipd.display(ipd.Audio(data=y, rate=sr))
    print("Harmonic")
    ipd.display(ipd.Audio(data=y_h, rate=sr))
    print("Percussive")
    ipd.display(ipd.Audio(data=y_p, rate=sr))

In [ ]:
def mel_spec_hpss(y, y_h, y_p, sr):
    fig = pl.figure(figsize=(12,4))
    titles = ('Origional', 'Harmonic', 'Percussive')
    for i, x in enumerate ((y, y_h, y_p)):
        pl.subplot(1,3,i+1)
        M = librosa.feature.melspectrogram(y=x, n_fft=2048, hop_length=1024, n_mels=128)
        M_dB = librosa.amplitude_to_db(M)
        img = librosa.display.specshow(M_dB, x_axis='time', y_axis='mel', sr=sr, ax=fig.gca(), cmap='viridis')
        pl.title(titles[i])
    pl.show()

We will use the HPSS implementation provided by the **librosa** library.

In [ ]:
# (1) Let us start with the piano phrase
y, sr = librosa.load(fn_wav_list[0])
y_h, y_p = librosa.effects.hpss(y)

In [ ]:
# Let's observe the Mel spectrograms
mel_spec_hpss(y, y_h, y_p, sr)

In [ ]:
# Let's listen to the original and the separated voices
audio_playback_hpss(y, y_h, y_p, sr)

**Obervations**:
- which signal parts end up in the harmonic and the percussive stream?
- how to better separate the transients in the percussive parts?

**Solution**:
- modify the *margin* parameter (see for instance https://librosa.org/doc/main/generated/librosa.decompose.hpss.html)

In [ ]:
y_h, y_p = librosa.effects.hpss(y, margin=(1, 10))

In [ ]:
mel_spec_hpss(y, y_h, y_p, sr)
audio_playback_hpss(y, y_h, y_p, sr)

Now there are less harmonic components audible in the percussive stream

### Example 2: Drum beat

In [ ]:
y, sr = librosa.load(fn_wav_list[1])
y_h, y_p = librosa.effects.hpss(y)
mel_spec_hpss(y, y_h, y_p, sr)
audio_playback_hpss(y, y_h, y_p, sr)

**Observations**:
- Which instruments end up in the harmonic stream?
- Which stream do you think is a better starting point for a rhythmic analysis (tempo estimation for instance)?

### Example 3: Melodic percussive instruments

Let us analyze a third recording. You can hear pitched drum instruments.
After the HPSS is applied, the underlying "melody" is better audible.

In [ ]:
y, sr = librosa.load(fn_wav_list[2])
y_h, y_p = librosa.effects.hpss(y)
mel_spec_hpss(y, y_h, y_p, sr)
audio_playback_hpss(y, y_h, y_p, sr)

## Non-Negative Matrix Factorization

We will try to use NMF to decompose the first 4 seconds of the drum beat into its main components (kick drum, snare drum etc.). So we'll use 5 components to start with.

In [ ]:
y, sr = librosa.load(fn_wav_list[1], duration=4)
S = np.abs(librosa.stft(y))
comps, acts = librosa.decompose.decompose(S, n_components=5, sort=True)

In [ ]:
import matplotlib.pyplot as plt
layout = [list(".AAAA"), list("BCCCC"), list(".DDDD")]
fig, ax = plt.subplot_mosaic(layout, constrained_layout=True)
librosa.display.specshow(librosa.amplitude_to_db(S, ref=np.max),
                         y_axis='log', x_axis='time', ax=ax['A'])
ax['A'].set(title='Input spectrogram')
ax['A'].label_outer()
librosa.display.specshow(librosa.amplitude_to_db(comps,
                                                 ref=np.max),
                         y_axis='log', ax=ax['B'])
ax['B'].set(title='Components')
ax['B'].label_outer()
ax['B'].sharey(ax['A'])
librosa.display.specshow(acts, x_axis='time', ax=ax['C'], cmap='gray_r')
ax['C'].set(ylabel='Components', title='Activations')
ax['C'].sharex(ax['A'])
ax['C'].label_outer()

# reconstruct the spectrogram with the estimated spectral basis functions and activations
S_approx = comps.dot(acts)

img = librosa.display.specshow(librosa.amplitude_to_db(S_approx,
                                                       ref=np.max),
                               y_axis='log', x_axis='time', ax=ax['D'])
ax['D'].set(title='Reconstructed spectrogram')
ax['D'].sharex(ax['A'])
ax['D'].sharey(ax['A'])
ax['D'].label_outer()
fig.colorbar(img, ax=list(ax.values()), format="%+2.f dB")

ipd.display(ipd.Audio(data=y, rate=sr))

**Observations** 
- The upper component captures the snare drum (hits around 0.25s, 1.3s, etc.)
- The kick drum (hits around 0s, 0.5s, 1.1s, etc...) is captured by multiple components (3-5).
- Any idea how to get a better decomposition?

We will continue here: https://www.audiolabs-erlangen.de/resources/MIR/FMP/C8/C8S3_NMFSpecFac.html